In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# GPU setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using: {device} — {torch.cuda.get_device_name(0)}")

# Path to your CSVs
DATA = r"C:\Users\Sobia Khan\Downloads\DATA-adni\ADNIMERGE_CSVs"
print(f"✅ Data folder: {DATA}")

✅ Using: cuda — NVIDIA GeForce RTX 3060
✅ Data folder: C:\Users\Sobia Khan\Downloads\DATA-adni\ADNIMERGE_CSVs


In [2]:
# Use DXSUM as our starting point instead of ADNIMERGE
dx = pd.read_csv(os.path.join(DATA, "DXSUM.csv"), low_memory=False)
registry = pd.read_csv(os.path.join(DATA, "REGISTRY.csv"), low_memory=False)
ptdemog = pd.read_csv(os.path.join(DATA, "PTDEMOG.csv"), low_memory=False)

print(f"DXSUM shape: {dx.shape}")
print(f"REGISTRY shape: {registry.shape}")
print(f"PTDEMOG shape: {ptdemog.shape}")
print(f"\nDiagnosis counts:\n{dx['DIAGNOSIS'].value_counts()}")

DXSUM shape: (15881, 42)
REGISTRY shape: (28858, 27)
PTDEMOG shape: (6219, 85)

Diagnosis counts:
DIAGNOSIS
MCI         6565
CN          6275
Dementia    2996
Name: count, dtype: int64


In [3]:
#Load all the tabels
# Load all your CSV files
mmse    = pd.read_csv(os.path.join(DATA, "MMSE.csv"), low_memory=False)
adas    = pd.read_csv(os.path.join(DATA, "ADAS.csv"), low_memory=False)
cdr     = pd.read_csv(os.path.join(DATA, "CDR.csv"), low_memory=False)
faq     = pd.read_csv(os.path.join(DATA, "FAQ.csv"), low_memory=False)
apoe    = pd.read_csv(os.path.join(DATA, "APOERES.csv"), low_memory=False)
mri     = pd.read_csv(os.path.join(DATA, "UCSFFSX7.csv"), low_memory=False)
fdg     = pd.read_csv(os.path.join(DATA, "UCBERKELEYFDG_8mm.csv"), low_memory=False)
amy     = pd.read_csv(os.path.join(DATA, "UCBERKELEY_AMY_6MM.csv"), low_memory=False)
csf     = pd.read_csv(os.path.join(DATA, "UPENNBIOMK_MASTER.csv"), low_memory=False)
neurobat= pd.read_csv(os.path.join(DATA, "NEUROBAT.csv"), low_memory=False)

print("✅ All files loaded!")
print(f"  MMSE:     {mmse.shape}")
print(f"  ADAS:     {adas.shape}")
print(f"  CDR:      {cdr.shape}")
print(f"  FAQ:      {faq.shape}")
print(f"  APOE:     {apoe.shape}")
print(f"  MRI:      {mri.shape}")
print(f"  FDG-PET:  {fdg.shape}")
print(f"  Amy-PET:  {amy.shape}")
print(f"  CSF:      {csf.shape}")
print(f"  NEUROBAT: {neurobat.shape}")

✅ All files loaded!
  MMSE:     (14599, 59)
  ADAS:     (12868, 17)
  CDR:      (14617, 26)
  FAQ:      (13272, 28)
  APOE:     (3008, 17)
  MRI:      (12151, 348)
  FDG-PET:  (7524, 11)
  Amy-PET:  (4582, 345)
  CSF:      (5876, 15)
  NEUROBAT: (17622, 84)


In [4]:
#Build baseline master table:
# Start with REGISTRY as backbone — one row per subject per visit
# Get baseline visit only (VISCODE == 'bl')
reg_bl = registry[registry['VISCODE'] == 'bl'].copy()
print(f"Baseline subjects in REGISTRY: {len(reg_bl)}")

# Add diagnosis at baseline
dx_bl = dx[dx['VISCODE2'] == 'bl'][['RID','DIAGNOSIS']].drop_duplicates('RID')
dx_bl.columns = ['RID','DX_bl']

# Merge
master = reg_bl[['RID','VISCODE','SITEID','USERDATE']].merge(
    dx_bl, on='RID', how='left')

# Add demographics
demog_cols = ['RID','PTGENDER','PTDOB','PTEDUCAT','PTETHCAT','PTRACCAT','PTMARRY']
demog_cols = [c for c in demog_cols if c in ptdemog.columns]
master = master.merge(ptdemog[demog_cols].drop_duplicates('RID'), 
                      on='RID', how='left')

print(f"\nMaster table shape: {master.shape}")
print(f"\nBaseline diagnosis distribution:")
print(master['DX_bl'].value_counts())

Baseline subjects in REGISTRY: 1663

Master table shape: (1663, 11)

Baseline diagnosis distribution:
DX_bl
MCI         767
CN          608
Dementia    266
Name: count, dtype: int64


In [5]:
# Check what GENOTYPE looks like
print("Sample GENOTYPE values:")
print(apoe['GENOTYPE'].value_counts().head(10))

Sample GENOTYPE values:
GENOTYPE
3/3    1417
3/4    1016
4/4     275
2/3     219
2/4      71
2/2      10
Name: count, dtype: int64


In [6]:
# GENOTYPE is stored as "3/4", "4/4", "3/3" etc.
apoe_clean = apoe[['RID','GENOTYPE']].drop_duplicates('RID').copy()

# Split "3/4" into two alleles
apoe_clean['APGEN1'] = apoe_clean['GENOTYPE'].str.split('/').str[0].astype(float)
apoe_clean['APGEN2'] = apoe_clean['GENOTYPE'].str.split('/').str[1].astype(float)

# APOE4 carrier = has at least one e4 allele
apoe_clean['APOE4'] = ((apoe_clean['APGEN1'] == 4) | 
                        (apoe_clean['APGEN2'] == 4)).astype(int)

# APOE4 dose: 0 = no e4, 1 = one e4, 2 = two e4 (homozygous)
apoe_clean['APOE4_count'] = (apoe_clean['APGEN1'] == 4).astype(int) + \
                              (apoe_clean['APGEN2'] == 4).astype(int)

master = master.merge(apoe_clean[['RID','APOE4','APOE4_count']], 
                      on='RID', how='left')

print(f"APOE4 carrier rate:")
print(master['APOE4'].value_counts())
print(f"\nAPOE4 by diagnosis:")
print(master.groupby('DX_bl')['APOE4'].mean().round(3))

APOE4 carrier rate:
APOE4
0.0    897
1.0    731
Name: count, dtype: int64

APOE4 by diagnosis:
DX_bl
CN          0.319
Dementia    0.665
MCI         0.473
Name: APOE4, dtype: float64


In [7]:
# Check actual VISCODE values in each table
print("MMSE VISCODE values:", mmse['VISCODE'].unique()[:10])
print("CDR VISCODE values:",  cdr['VISCODE'].unique()[:10])
print("REGISTRY VISCODE:",    registry['VISCODE'].unique()[:10])

MMSE VISCODE values: ['sc' 'f' 'm06' 'm12' 'm18' 'm36' 'm24' 'm48' 'm60' 'm72']
CDR VISCODE values: ['sc' 'f' 'm06' 'm12' 'm36' 'm18' 'uns1' 'm24' 'm48' 'm60']
REGISTRY VISCODE: ['sc' 'f' 'bl' 'm06' 'm12' 'm18' 'm24' 'm30' 'm36' 'uns1']


In [8]:
# ADNI uses 'sc' for screening = baseline cognitive tests
# REGISTRY uses 'bl' for baseline

# MMSE — use 'sc' visit
mmse_bl = mmse[mmse['VISCODE'] == 'sc'][['RID','MMSCORE']].drop_duplicates('RID')

# CDR — use 'sc' visit
cdr_bl = cdr[cdr['VISCODE'] == 'sc'][['RID','CDGLOBAL','CDRSB']].drop_duplicates('RID')

# FAQ — check its viscode first
print("FAQ VISCODE values:", faq['VISCODE'].unique()[:8])
faq_bl = faq[faq['VISCODE'].isin(['sc','bl'])][['RID','FAQTOTAL']].drop_duplicates('RID')

# ADAS — check too
print("ADAS VISCODE values:", adas['VISCODE'].unique()[:8])
adas_bl = adas[adas['VISCODE'].isin(['sc','bl'])][['RID','TOTAL13']].drop_duplicates('RID')

# Rebuild master from scratch cleanly
reg_bl = registry[registry['VISCODE'] == 'bl'].copy()
dx_bl  = dx[dx['VISCODE2'] == 'bl'][['RID','DIAGNOSIS']].drop_duplicates('RID')
dx_bl.columns = ['RID','DX_bl']

master = reg_bl[['RID','VISCODE','SITEID']].merge(dx_bl, on='RID', how='left')
master = master.merge(apoe_clean[['RID','APOE4','APOE4_count']], on='RID', how='left')
master = master.merge(mmse_bl, on='RID', how='left')
master = master.merge(cdr_bl,  on='RID', how='left')
master = master.merge(faq_bl,  on='RID', how='left')
master = master.merge(adas_bl, on='RID', how='left')

# Add age from PTDEMOG
if 'AGE' in ptdemog.columns:
    age_col = ptdemog[['RID','AGE']].drop_duplicates('RID')
elif 'PTAGE' in ptdemog.columns:
    age_col = ptdemog[['RID','PTAGE']].drop_duplicates('RID')
    age_col.columns = ['RID','AGE']
else:
    # Calculate from birth year
    print("Age columns available:", [c for c in ptdemog.columns if 'AGE' in c.upper() or 'DOB' in c.upper()])
    age_col = None

if age_col is not None:
    master = master.merge(age_col, on='RID', how='left')

# Add education and gender
demog_extra = []
for col in ['PTEDUCAT','PTGENDER','PTETHCAT','PTRACCAT']:
    if col in ptdemog.columns:
        demog_extra.append(col)

master = master.merge(
    ptdemog[['RID'] + demog_extra].drop_duplicates('RID'),
    on='RID', how='left')

print(f"✅ Master table shape: {master.shape}")
print(f"\nDiagnosis distribution:")
print(master['DX_bl'].value_counts())
print(f"\nMean MMSE by diagnosis:")
print(master.groupby('DX_bl')['MMSCORE'].mean().round(2))
print(f"\nMean CDR-SB by diagnosis:")
print(master.groupby('DX_bl')['CDRSB'].mean().round(2))
print(f"\nMean ADAS-13 by diagnosis:")
print(master.groupby('DX_bl')['TOTAL13'].mean().round(2))
print(f"\nAPOE4 rate by diagnosis:")
print(master.groupby('DX_bl')['APOE4'].mean().round(3))

FAQ VISCODE values: ['bl' 'm06' 'm12' 'm18' 'uns1' 'm24' 'm36' 'm48']
ADAS VISCODE values: ['m12' 'm06' 'bl' 'm24' 'm36' 'm18' 'm48' 'uns1']
Age columns available: ['PTDOB', 'PTDOBYY', 'PTENGSPKAGE', 'PTIMMAGE', 'LANGUAGE_CODE']
✅ Master table shape: (1663, 15)

Diagnosis distribution:
DX_bl
MCI         767
CN          608
Dementia    266
Name: count, dtype: int64

Mean MMSE by diagnosis:
DX_bl
CN          29.12
Dementia    23.22
MCI         27.47
Name: MMSCORE, dtype: float64

Mean CDR-SB by diagnosis:
DX_bl
CN          0.04
Dementia    4.33
MCI         1.51
Name: CDRSB, dtype: float64

Mean ADAS-13 by diagnosis:
DX_bl
CN           8.67
Dementia    29.19
MCI         16.56
Name: TOTAL13, dtype: float64

APOE4 rate by diagnosis:
DX_bl
CN          0.319
Dementia    0.665
MCI         0.473
Name: APOE4, dtype: float64


In [9]:
# Check what columns exist in MRI file
print("All MRI columns:")
print(mri.columns.tolist())
print(f"\nMRI VISCODE values: {mri['VISCODE'].unique()[:10]}")
print(f"\nMRI shape: {mri.shape}")

All MRI columns:
['ORIGPROT', 'COLPROT', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'IMAGEUID', 'FIELD_STRENGTH', 'EXAMDATE', 'RUNDATE', 'STATUS', 'FSVER', 'OVERALLQC', 'TEMPQC', 'FRONTQC', 'PARQC', 'INSULAQC', 'OCCQC', 'BGQC', 'CWMQC', 'VENTQC', 'HIPPOQC', 'ST101SV', 'ST102CV', 'ST102SA', 'ST102TA', 'ST102TS', 'ST103CV', 'ST103SA', 'ST103TA', 'ST103TS', 'ST104CV', 'ST104SA', 'ST104TA', 'ST104TS', 'ST105CV', 'ST105SA', 'ST105TA', 'ST105TS', 'ST106CV', 'ST106SA', 'ST106TA', 'ST106TS', 'ST107CV', 'ST107SA', 'ST107TA', 'ST107TS', 'ST108CV', 'ST108SA', 'ST108TA', 'ST108TS', 'ST109CV', 'ST109SA', 'ST109TA', 'ST109TS', 'ST10CV', 'ST110CV', 'ST110SA', 'ST110TA', 'ST110TS', 'ST111CV', 'ST111SA', 'ST111TA', 'ST111TS', 'ST112SV', 'ST113CV', 'ST113SA', 'ST113TA', 'ST113TS', 'ST114CV', 'ST114SA', 'ST114TA', 'ST114TS', 'ST115CV', 'ST115SA', 'ST115TA', 'ST115TS', 'ST116CV', 'ST116SA', 'ST116TA', 'ST116TS', 'ST117CV', 'ST117SA', 'ST117TA', 'ST117TS', 'ST118CV', 'ST118SA', 'ST118TA', 'ST118TS', 'ST119CV', 

In [10]:
# Key MRI columns from UCSFFSX7
# ST29SV = Left Hippocampus, ST88SV = Right Hippocampus
# ST40TS = Left Entorhinal thickness, ST99TS = Right Entorhinal thickness  
# ST101SV = ICV (intracranial volume)

# Skip QC filter — use 'sc' as baseline visit
mri_cols_needed = ['RID', 'VISCODE', 'FIELD_STRENGTH',
                   'ST29SV',   # Left Hippocampus volume
                   'ST88SV',   # Right Hippocampus volume
                   'ST40TS',   # Left Entorhinal thickness
                   'ST99TS',   # Right Entorhinal thickness
                   'ST101SV',  # ICV
                   ]

mri_cols_needed = [c for c in mri_cols_needed if c in mri.columns]

# Use 'sc' for baseline — it has 1606 entries
mri_bl = mri[mri['VISCODE'] == 'sc'][mri_cols_needed].copy()
mri_bl = mri_bl.drop_duplicates('RID')
print(f"MRI baseline subjects: {len(mri_bl)}")

# Check if values are actually present
print(f"\nST29SV (Left Hippo) sample values:")
print(mri_bl['ST29SV'].dropna().head(5).values)
print(f"ST29SV missing: {mri_bl['ST29SV'].isna().sum()} / {len(mri_bl)}")

# Calculate hippocampal volumes
mri_bl['HIPPO_TOTAL'] = mri_bl['ST29SV'] + mri_bl['ST88SV']
mri_bl['HIPPO_ICV']   = mri_bl['HIPPO_TOTAL'] / mri_bl['ST101SV'] * 100

# Rebuild master WITHOUT old MRI columns first
mri_drop_cols = ['FIELD_STRENGTH','OVERALLQC','ST29SV','ST88SV',
                 'ST40TS','ST99TS','ST101SV','HIPPO_TOTAL','HIPPO_ICV']
master = master.drop(columns=[c for c in mri_drop_cols if c in master.columns])

# Now merge fresh
master = master.merge(
    mri_bl.drop(columns=['VISCODE']),
    on='RID', how='left')

print(f"\nMaster shape: {master.shape}")
print(f"Subjects with MRI: {master['HIPPO_TOTAL'].notna().sum()}")
print(f"\nMean Hippocampal Volume by diagnosis:")
print(master.groupby('DX_bl')['HIPPO_TOTAL'].mean().round(1))
print(f"\nMean ICV-corrected Hippo by diagnosis:")
print(master.groupby('DX_bl')['HIPPO_ICV'].mean().round(4))

MRI baseline subjects: 1603

ST29SV (Left Hippo) sample values:
[4172.  2821.9 3199.9 3767.7 4017.6]
ST29SV missing: 0 / 1603

Master shape: (1663, 23)
Subjects with MRI: 1482

Mean Hippocampal Volume by diagnosis:
DX_bl
CN          7549.4
Dementia    5912.0
MCI         6844.1
Name: HIPPO_TOTAL, dtype: float64

Mean ICV-corrected Hippo by diagnosis:
DX_bl
CN          415.1384
Dementia    330.7769
MCI         372.5189
Name: HIPPO_ICV, dtype: float64


In [15]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

dx_order = ['CN', 'MCI', 'Dementia']
colors   = ['#2ecc71', '#f39c12', '#e74c3c']

dx_counts = master['DX_bl'].value_counts().reindex(dx_order)
apoe_dx   = master.groupby('DX_bl')['APOE4'].mean().reindex(dx_order)*100

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Subject Count', 'APOE4 Carrier Rate (%)',
        'Hippocampal Volume (mm³)', 'MMSE Distribution',
        'CDR-SB Distribution', 'Mean Scores by Diagnosis'
    ],
    vertical_spacing=0.18,
    horizontal_spacing=0.12
)

# 1. Subject counts — extend y range so labels fit
fig.add_trace(go.Bar(
    x=dx_order, y=dx_counts.values,
    marker_color=colors, showlegend=False,
    text=dx_counts.values, textposition='inside',  # inside = never cut off
    textfont=dict(size=14, color='white')
), row=1, col=1)
fig.update_yaxes(range=[0, dx_counts.max()*1.2], row=1, col=1)

# 2. APOE4 rate — same fix
fig.add_trace(go.Bar(
    x=dx_order, y=apoe_dx.values,
    marker_color=colors, showlegend=False,
    text=[f'{v:.1f}%' for v in apoe_dx.values],
    textposition='inside',
    textfont=dict(size=14, color='white')
), row=1, col=2)
fig.update_yaxes(range=[0, 85], row=1, col=2)

# 3. Hippocampal volume box
for dx, color in zip(dx_order, colors):
    vals = master[master['DX_bl']==dx]['HIPPO_TOTAL'].dropna().values
    fig.add_trace(go.Box(
        y=vals, name=dx, marker_color=color, showlegend=True
    ), row=1, col=3)

# 4. MMSE histogram
for dx, color in zip(dx_order, colors):
    vals = master[master['DX_bl']==dx]['MMSCORE'].dropna().values
    fig.add_trace(go.Histogram(
        x=vals, name=dx, marker_color=color,
        opacity=0.65, showlegend=False, nbinsx=12
    ), row=2, col=1)

# 5. CDR-SB histogram
for dx, color in zip(dx_order, colors):
    vals = master[master['DX_bl']==dx]['CDRSB'].dropna().values
    fig.add_trace(go.Histogram(
        x=vals, name=dx, marker_color=color,
        opacity=0.65, showlegend=False, nbinsx=12
    ), row=2, col=2)

# 6. Mean scores grouped bar
metrics = ['MMSE', 'CDR-SB x10', 'APOE4 %', 'Hippo /100']
for dx, color in zip(dx_order, colors):
    sub = master[master['DX_bl']==dx]
    vals = [
        sub['MMSCORE'].mean(),
        sub['CDRSB'].mean() * 10,
        sub['APOE4'].mean() * 100,
        sub['HIPPO_TOTAL'].mean() / 100
    ]
    fig.add_trace(go.Bar(
        name=dx, x=metrics, y=vals,
        marker_color=color, showlegend=False
    ), row=2, col=3)

fig.update_layout(
    height=750, width=1200,
    title_text='ADNI Baseline Dataset — Exploratory Analysis',
    title_font_size=16,
    title_x=0.5,
    barmode='group',
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=80, b=60, l=60, r=60),
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='right', x=1
    )
)

# Clean grid styling
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#eeeeee')

fig.write_html('01_baseline_overview.html')
print("✅ Saved: 01_baseline_overview.html")
fig.show()

✅ Saved: 01_baseline_overview.html


In [16]:
# Save master table
master.to_csv(
    r'C:\Users\Sobia Khan\Downloads\DATA-adni\ADNIMERGE_CSVs\master_baseline.csv', 
    index=False)
print(f"✅ Saved master_baseline.csv — {master.shape[0]} subjects × {master.shape[1]} features")

✅ Saved master_baseline.csv — 1663 subjects × 23 features
